In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time as time 
import scipy.io as sio

pi = np.pi
sin = np.sin
cos = np.cos
ln = np.log
exp = np.exp
tanh = np.tanh
cosh = np.cosh
linspace = np.linspace
sqrt = np.sqrt
fft = np.fft.fft
ifft = np.fft.ifft
fft2 = np.fft.fft2
ifft2 = np.fft.ifft2
mean = np.mean
real = np.real

def sech(x):
    return 1/cosh(x)

In [ ]:
# -------------------------------
# ETDRK4 Algorithm
# -------------------------------
def ETDRK4F2D(xl, xr, yl, yr, Nx, Ny, T, dt, u0, beta, epsilon, lam):
    N = Nx * Ny
    M = int(T/dt)
    omx = xr - xl
    omy = yr - yl
    lx = 2 * pi / omx
    ly = 2 * pi / omy
    x = linspace(xl, xr, Nx, endpoint=False)
    y = linspace(yl, yr, Ny, endpoint=False)
    X, Y = np.meshgrid(x, y)
    kx = np.fft.fftfreq(Nx, 1/(Nx * lx))
    kx[0] = 1
    ky = np.fft.fftfreq(Ny, 1/(Ny * ly))
    Kx, Ky = np.meshgrid(kx, ky)
    Kx = np.reshape(Kx, (1, N))
    Ky = np.reshape(Ky, (1, N))
    L = 1j * (epsilon ** 2 * Kx ** 3 - lam * Ky ** 2/Kx)
    for m in range(Ny):
        L[0,m*Nx] = 0 
        Kx[0, m*Nx] = 0
    E = exp(dt * L)
    Eh = exp(dt * L/2)
    nr = 64
    r = exp(1j * pi * linspace(1, 2*nr-1, nr)/nr)
    Lt =  dt * np.tile(L,(nr,1))
    LR = np.transpose(Lt) + np.tile(r, (N,1))
    Q = dt *  mean( (exp(LR/2)-1)/LR, 1) 
    f1 = dt *  mean( (-4 - LR + exp(LR)*(4 - 3*LR + LR ** 2))/(LR ** 3), 1)
    f2 = dt *  mean( (2 + LR + exp(LR) * (-2 + LR))/(LR ** 3), 1) 
    f3 = dt *  mean( (-4 -3 * LR - LR ** 2 + exp(LR)  * (4-LR))/(LR ** 3), 1)
    g = -0.5 * 1j * beta * Kx
    u1=u0(X, Y)
    v = fft2(u1)
    t = 0 
    for m in range(M):
        Nv = fft2(real(ifft2(v)) ** 2)
        Nv = g * Nv.reshape(1, N)
        v = v.reshape(1, N)
        a = Eh * v + Q * Nv
        aa = a.reshape(Ny, Nx)
        Na = fft2(real(ifft2(aa)) ** 2)
        Na = g * Na.reshape(1, N)
        b = Eh * v + Q * Na
        b = b.reshape(Ny, Nx)
        Nb = fft2(real(ifft2(b)) ** 2)
        Nb = g * Nb.reshape(1, N)
        c = Eh * a  + Q * (2 * Nb - Nv)
        c = c.reshape(Ny, Nx)
        Nc = fft2(real(ifft2(c)) ** 2)
        Nc = g * Nc.reshape(1, N)
        v = E * v + Nv * f1 + 2*(Na + Nb) * f2 + Nc * f3
        v = v.reshape(Ny, Nx)
        t += dt
        u =real(ifft2(v))
    return x, y, u, u1

In [ ]:
# -------------------------------
# Parameter setup
# -------------------------------
eps = 1e-100
xl = -pi
xr = -xl
yl = -pi 
yr = -yl
Nx = 2 ** 9
Ny = 2 ** 9
T = 0.3
M=1000
dt = T/M
beta = 1
epsilon = 0.02
lam = -1
size=10;
# rd=np.random.rand(1)
def u0(x, y):
    r = sqrt(x ** 2 + y ** 2)
    constant1=1.5+np.random.rand(1)
    constant2=1.5+np.random.rand(1)
    return 8 * x * tanh(constant1*r) /(r+ eps) * sech(constant2*r) ** 2 

In [ ]:
# -------------------------------
# Data generation
# -------------------------------
time=np.zeros((int(T/dt)+1,size));
x=np.zeros((Nx,size));
y=np.zeros((Ny,size));
u_end=np.zeros((Nx,Ny,size));
u1=np.zeros((Nx,Ny,size));
for i in range(size):
   x[:,i], y[:,i], u_end[:,:,i],u1[:,:,i]  = ETDRK4F2D(xl, xr, yl, yr, Nx, Ny, T, dt, u0, beta, epsilon, lam)

In [ ]:
# -------------------------------
# Save dataset
# -------------------------------
np.save(f"u_end_{Nx}.npy",u_end)
np.save(f"u1_{Nx}.npy",u1)